# Policy Training with MuJoCo Warp

This notebook provides a skeleton for training a reinforcement learning policy using the `stretch_mujoco_warp` model and MuJoCo Warp in Python. MuJoCo Warp (MJWarp) enables high-throughput, parallelized physics simulations directly on NVIDIA GPUs, drastically reducing RL model training time.

In [1]:
!uv pip install mujoco-warp
!uv pip install -U "jax[cuda12]" optax flax
!uv pip install numpy==2.2.5


Using Python 3.12.12 environment at: /home/shehab/Desktop/stretch4_mujoco/.venv
⠋ Resolving dependencies...                                                     
⠋ Resolving dependencies...                                                     
⠙ Resolving dependencies...                                                     
⠙ mujoco-warp==3.8.0.2                                                          
⠙ absl-py==2.4.0                                                                
⠙ etils==1.14.0                                                                 
⠙ etils==1.14.0                                                                 
⠙ mujoco==3.8.0                                                                 
⠙ numpy==2.2.5                                                                  
⠙ warp-lang==1.13.0                                                             
⠙ fsspec==2026.4.0                                                              
⠙ typing-extensions==4.15.0  

Using Python 3.12.12 environment at: /home/shehab/Desktop/stretch4_mujoco/.venv
⠋ Resolving dependencies...                                                     
⠋ Resolving dependencies...                                                     
⠙ Resolving dependencies...                                                     


⠙ jax==0.10.0                                                                   
⠙ jax==0.10.0                                                                   
⠙ optax==0.2.8                                                                  
⠙ flax==0.12.7                                                                  
⠙ jaxlib==0.10.0                                                                
⠙ ml-dtypes==0.5.4                                                              
⠙ numpy==2.4.4                                                                  
⠙ numpy==2.4.4                                                                  
⠙ numpy==2.4.4                                                                  
⠙ numpy==2.4.4                                                                  
⠙ opt-einsum==3.4.0                                                             
⠙ scipy==1.17.1                                                                 
⠙ jax-cuda12-plugin==0.10.0


⠙ jax-cuda12-pjrt==0.10.0                                                       
⠙ absl-py==2.4.0                                                                
⠙ msgpack==1.1.2                                                                
⠙ orbax-checkpoint==0.11.39                                                     
⠙ tensorstore==0.1.83                                                           
⠙ rich==15.0.0                                                                  
⠙ simplejson==4.1.1                                                             


Resolved 43 packages in 176ms
⠋ Preparing packages... (0/0)                                                   
⠋ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)                                                   
⠙  (1/1)                                                                        
Prepared 1 package in 0.25ms
Uninstalled 1 package in 4ms
░░░░░░░░░░░░░░░░░░░░ [0/0] Installing wheels...                                 
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 
░░░░░░░░░░░░░░░░░░░░ [0/1] numpy==2.4.4                                         
████████████████████ [1/1] numpy==2.4.4                                         
Installed 1 package in 10ms
 - numpy==2.2.5
 + numpy==2.4.4


Using Python 3.12.12 environment at: /home/shehab/Desktop/stretch4_mujoco/.venv
⠋ Resolving dependencies...                                                     
⠙ Resolving dependencies...                                                     
⠋ Resolving dependencies...                                                     
⠙ Resolving dependencies...                                                     
⠙ numpy==2.2.5                                                                  
⠙                                                                               
Resolved 1 package in 6ms
Uninstalled 1 package in 17ms
░░░░░░░░░░░░░░░░░░░░ [0/0] Installing wheels...                                 
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 


░░░░░░░░░░░░░░░░░░░░ [0/1] numpy==2.2.5                                         
████████████████████ [1/1] numpy==2.2.5                                         
Installed 1 package in 48ms
 - numpy==2.4.4
 + numpy==2.2.5


In [11]:
import os
import subprocess
# import mediapy as media
import numpy as np
import warp as wp

# Set up GPU rendering.
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Configuring MuJoCo for GPU rendering, setting ', end="")
%env MUJOCO_GL=egl

# avoid warp output in cells
wp.config.quiet = True

Thu May  7 21:20:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:01:00.0  On |                  Off |
|  0%   32C    P8             45W /  480W |    3096MiB /  24564MiB |     13%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Load the Stretch Environment
We dynamically generate the MJCF using `Stretch4MujocoSimulator.get_robot_xml_path()`, which generates the URDF and parses it. It then creates the `stretch_4_dynamic.xml` which replaces the `.STL` collision meshes with solid primitives for efficient GPU evaluation. We then load the default scene that includes the dynamically generated robot XML.

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import mujoco
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import warp as wp
import numpy as np

try:
    import mujoco_warp as mjw
except ImportError:
    print("mujoco_warp not installed!")

wp.init()

from stretch_mujoco.robocasa_gen import model_generation_wizard
from stretch_mujoco.stretch4_mujoco_simulator import Stretch4MujocoSimulator

# 1. Generate the robot XML dynamically
robot_xml_path = Stretch4MujocoSimulator.get_robot_xml_path()

# 2. Use Robocasa to generate the scene
model, xml, objects_info = model_generation_wizard(
    stretch_xml_absolute=robot_xml_path,
    layout=1,
    style=1,
    task="PickPlaceCounterToCabinet"
)

# 3. Disable unsupported solver options
model.opt.noslip_iterations = 0
model.opt.integrator = mujoco.mjtIntegrator.mjINT_EULER

# Parse into MjModel
warp_model = mjw.put_model(model)
print("Model loaded successfully. Nu:", model.nu, "Nq:", model.nq, "Nv:", model.nv)

NWORLD = 128
print(f"Initializing {NWORLD} environments for training.")
# Pass nconmax and njmax manually just to be safe
data = mjw.make_data(model, nworld=NWORLD, nconmax=512, njmax=512)
mjw.forward(warp_model, data)

global_mj_model = model
global_warp_model = warp_model
global_warp_data = data

# Define the PPO Networks and Loss Functions

def sample_normal(rng, mean, log_std):
    std = jnp.exp(log_std)
    return mean + std * jax.random.normal(rng, mean.shape)

def log_prob_normal(x, mean, log_std):
    std = jnp.exp(log_std)
    var = std ** 2
    log_scale = log_std + 0.5 * jnp.log(2.0 * jnp.pi)
    return -((x - mean) ** 2) / (2.0 * var) - log_scale

class ActorCritic(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, x):
        # Actor
        a = nn.Dense(256)(x)
        a = nn.relu(a)
        a = nn.Dense(256)(a)
        a = nn.relu(a)
        actor_mean = nn.Dense(self.action_dim)(a)
        
        actor_log_std = self.param("log_std", nn.initializers.zeros, (self.action_dim,))
        
        # Critic
        c = nn.Dense(256)(x)
        c = nn.relu(c)
        c = nn.Dense(256)(c)
        c = nn.relu(c)
        critic = nn.Dense(1)(c)
        
        return actor_mean, actor_log_std, jnp.squeeze(critic, -1)

@jax.jit
def get_action_and_value(params, x, rng):
    mean, log_std, value = ActorCritic(global_mj_model.nu).apply(params, x)
    action = sample_normal(rng, mean, log_std)
    log_prob = log_prob_normal(action, mean, log_std).sum(-1)
    return action, log_prob, value

@jax.jit
def get_value(params, x):
    _, _, value = ActorCritic(global_mj_model.nu).apply(params, x)
    return value

@jax.jit
def compute_gae(rewards, values, next_value, dones, gamma=0.99, gae_lambda=0.95):
    # Calculate GAE for a single rollout trajectory
    advantages = jnp.zeros_like(rewards)
    lastgaelam = jnp.zeros_like(rewards[0])
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            nextnonterminal = 1.0 - dones[t]
            nextvalues = next_value
        else:
            nextnonterminal = 1.0 - dones[t]
            nextvalues = values[t + 1]
        delta = rewards[t] + gamma * nextvalues * nextnonterminal - values[t]
        lastgaelam = delta + gamma * gae_lambda * nextnonterminal * lastgaelam
        advantages = advantages.at[t].set(lastgaelam)
    returns = advantages + values
    return advantages, returns

@jax.jit
def update_ppo(params, opt_state, obs, actions, log_probs_old, returns, advantages):
    def loss_fn(p):
        mean, log_std, value = ActorCritic(global_mj_model.nu).apply(p, obs)
        log_probs = log_prob_normal(actions, mean, log_std).sum(-1)
        
        ratio = jnp.exp(log_probs - log_probs_old)
        surr1 = ratio * advantages
        surr2 = jnp.clip(ratio, 1.0 - 0.2, 1.0 + 0.2) * advantages
        
        actor_loss = -jnp.minimum(surr1, surr2).mean()
        critic_loss = jnp.mean((returns - value) ** 2)
        entropy_loss = jnp.mean(log_std + 0.5 + 0.5 * jnp.log(2 * jnp.pi))
        
        total_loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy_loss
        return total_loss, (actor_loss, critic_loss, entropy_loss)
    
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, (al, cl, el)), grads = grad_fn(params)
    updates, opt_state = tx.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, opt_state, loss

def get_obs_from_warp(global_warp_data):
    # observation = concat [qpos, qvel]
    qpos = wp.to_jax(global_warp_data.qpos)
    qvel = wp.to_jax(global_warp_data.qvel)
    return jnp.concatenate([qpos, qvel], axis=-1)

def compute_rewards(obs):
    alive_bonus = 1.0
    vel_penalty = -0.01 * jnp.sum(jnp.square(obs[:, global_mj_model.nq:]), axis=-1)
    return alive_bonus + vel_penalty

def env_step(jax_actions, global_warp_data, rng):
    wp_actions = wp.from_jax(jax_actions, dtype=wp.float32)
    global_warp_data.ctrl = wp_actions
    for _ in range(4):
        mjw.step(global_warp_model, global_warp_data)
    obs = get_obs_from_warp(global_warp_data)
    rewards = compute_rewards(obs)
    rng, reset_rng = jax.random.split(rng)
    resets = jax.random.bernoulli(reset_rng, p=0.01, shape=(NWORLD,))
    return obs, rewards, resets, rng

rng = jax.random.PRNGKey(42)
rng, init_rng = jax.random.split(rng)
dummy_obs = jnp.zeros((NWORLD, global_mj_model.nq + global_mj_model.nv))
params = ActorCritic(global_mj_model.nu).init(init_rng, dummy_obs)

tx = optax.adam(learning_rate=3e-4)
opt_state = tx.init(params)

num_iterations = 200 # VERY SMALL for testing
rollout_length = 1000 # VERY SMALL for testing
batch_size = NWORLD * rollout_length
minibatch_size = 256

print("Starting training...")
obs = get_obs_from_warp(global_warp_data)

for i in range(num_iterations):
    all_obs = []
    all_actions = []
    all_rewards = []
    all_dones = []
    all_log_probs = []
    all_values = []
    
    for step in range(rollout_length):
        rng, action_rng = jax.random.split(rng)
        actions, log_probs, values = get_action_and_value(params, obs, action_rng)
        
        next_obs, rewards, dones, rng = env_step(actions, global_warp_data, rng)
        
        all_obs.append(obs)
        all_actions.append(actions)
        all_rewards.append(rewards)
        all_dones.append(dones)
        all_log_probs.append(log_probs)
        all_values.append(values)
        
        obs = next_obs
        
    all_obs = jnp.stack(all_obs)
    all_actions = jnp.stack(all_actions)
    all_rewards = jnp.stack(all_rewards)
    all_dones = jnp.stack(all_dones)
    all_log_probs = jnp.stack(all_log_probs)
    all_values = jnp.stack(all_values)
    
    next_value = get_value(params, obs)
    advantages, returns = compute_gae(all_rewards, all_values, next_value, all_dones)
    
    b_obs = all_obs.reshape(-1, all_obs.shape[-1])
    b_actions = all_actions.reshape(-1, all_actions.shape[-1])
    b_log_probs = all_log_probs.reshape(-1)
    b_returns = returns.reshape(-1)
    b_advantages = advantages.reshape(-1)
    
    b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)
    
    params, opt_state, loss = update_ppo(
        params, opt_state, b_obs, b_actions, b_log_probs, b_returns, b_advantages
    )
    
    print(f"Iteration {i}, Mean Reward: {all_rewards.mean():.4f}, Loss: {loss:.4f}")
    
print("Training finished!")



[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)


[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly (e.g. pip install mink==0.0.5), otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


Generated URDF path: /tmp/tmpfvcw_k42/SE4_eames_eoa_wrist_dw4_tool_sg4_mujoco_stretch_4.urdf
Using urdf_path='/tmp/tmpfvcw_k42/SE4_eames_eoa_wrist_dw4_tool_sg4_mujoco_stretch_4.urdf'


[robosuite INFO] Loading controller configuration from: /home/shehab/Desktop/stretch4_mujoco/third_party/robosuite/robosuite/controllers/config/robots/default_pandaomron.json (composite_controller_factory.py:121)


DEFAULT XML: /home/shehab/Desktop/stretch4_mujoco/stretch_mujoco/models/stretch_4/stretch_4_dynamic.xml
Saving temp abs path xml: /home/shehab/Desktop/stretch4_mujoco/stretch_mujoco/models/stretch_temp_abs.xml
Initializing environment...
Showing configuration:
    Layout: One wall w/ island
    Style: Style001

Spawning environment...



...

/home/shehab/Desktop/stretch4_mujoco/.venv/lib/python3.12/site-packages/mujoco_warp/_src/io.py:170: UserWarning: pair 0: friction[0] (0.0) < MJ_MINMU (1e-05) with condim=3 may cause NaN
  warnings.warn(


Model loaded successfully. Nu: 10 Nq: 91 Nv: 87
Initializing 128 environments for training.


Starting training...


Iteration 0, Mean Reward: -128.5529, Loss: 1085117.5000


Iteration 1, Mean Reward: -227.3367, Loss: 3014388.2500
Training finished!


In [12]:
# Save the trained parameters
from flax import serialization

# Convert parameters to bytes
bytes_output = serialization.to_bytes(params)

# Save to disk
with open("trained_ppo_params.msgpack", "wb") as f:
    f.write(bytes_output)
    
print("Saved trained model parameters to trained_ppo_params.msgpack")


Saved trained model parameters to trained_ppo_params.msgpack


In [13]:
# Load the trained model and run it in the standard MuJoCo Viewer
import mujoco
import mujoco.viewer
import time
import jax.numpy as jnp
import numpy as np
from flax import serialization

# Load from disk
with open("trained_ppo_params.msgpack", "rb") as f:
    bytes_input = f.read()

loaded_params = serialization.from_bytes(params, bytes_input)

# Create a fresh MjData for standard simulation
mj_data = mujoco.MjData(global_mj_model)

# Reset environment
mujoco.mj_resetData(global_mj_model, mj_data)

print("Launching MuJoCo Viewer...")
try:
    with mujoco.viewer.launch_passive(global_mj_model, mj_data) as viewer:
        for step in range(1000):
            if not viewer.is_running():
                break
                
            # Construct observation
            obs = jnp.concatenate([jnp.array(mj_data.qpos), jnp.array(mj_data.qvel)], axis=-1)
            
            # Get deterministic action from the policy (mean of the distribution)
            action_mean, _, _ = ActorCritic(global_mj_model.nu).apply(loaded_params, obs)
            
            # Apply to simulation
            mj_data.ctrl[:] = np.array(action_mean)
            mujoco.mj_step(global_mj_model, mj_data)
            
            viewer.sync()
            time.sleep(global_mj_model.opt.timestep)
except Exception as e:
    print("Viewer closed or error:", e)


Launching MuJoCo Viewer...
